# Jacobian 引导调度的形式化权衡证书

这个 notebook 单独抽出调度实验背后的数学命题。实验看起来可能是经验性的，但只要接受局部指数衰减模型，决策规则本身有一个清楚的确定性证书。

## 设定

执行某个候选调度的短前缀后，假设我们测得两个量：

$$
L(a)>0,
\qquad
\lambda(a).
$$

这里 $L(a)$ 是前缀后的 loss，$\lambda(a)$ 是在到达状态处由 Jacobian 计算得到的局部有效收缩率。对剩余 horizon $H$、学习率 $\eta$ 和折扣 $\alpha$，定义

$$
c = 2\alpha\eta H,
\qquad c>0.
$$

局部价值模型预测

$$
\widehat L(a)
=
L(a)\exp\{-c\lambda(a)\}.
$$

这个公式包含两个相互竞争的项：

- $L(a)$ 衡量选择调度 $a$ 后的当前代价。
- $\lambda(a)$ 衡量预测的未来收缩能力。

核心问题是：一个当前 loss 更差的调度，什么时候仍然会被预测为终点更好？

## 定理

令 $a$ 和 $b$ 是两个候选调度。假设

$$
L(a)>0,\qquad L(b)>0,\qquad c>0.
$$

如果

$$
\log\frac{L(b)}{L(a)}
<
c\bigl(\lambda(b)-\lambda(a)\bigr),
$$

那么

$$
L(b)\exp\{-c\lambda(b)\}
<
L(a)\exp\{-c\lambda(a)\}.
$$

换句话说：候选 $b$ 的前缀 loss 可以比候选 $a$ 更大，但只要它的 Jacobian 收缩率优势足以偿还这个对数形式的前缀 loss 代价，它就会被预测为终点更优。

这就是 prefix-Jacobian 排序规则使用的精确权衡证书。

## 证明

把要证明的不等式两边同除以正数 $L(a)\exp\{-c\lambda(a)\}$。命题变为

$$
\frac{L(b)}{L(a)}
\exp\{-c(\lambda(b)-\lambda(a))\}
<1.
$$

因为对数函数在正数上保序，对两边取对数，等价于

$$
\log\frac{L(b)}{L(a)}
-c(\lambda(b)-\lambda(a))
<0.
$$

这正是定理假设。

## Lean4 形式化

下面的 Lean4 定理证明了同一个蕴含。它故意不依赖神经网络细节；网络和调度实验只负责提供实数 $L$ 和 $\lambda$。

**Lean 状态。** 下方 Lean 代码采用 Lean 4 core 风格，并已在远端服务器用 Lean 4.32.0 通过 `lean <file>.lean` 检查。


```lean
/-
Lean 4.32 core-verified log-domain tradeoff certificates.

The real-valued exponential predictor compares candidate b with baseline a:

  predicted_ratio = prefix_ratio * exp (- total_gain).

Taking logarithms gives the equivalent log-domain condition:

  log(predicted_ratio) = prefix_penalty - total_gain.

Thus predicted_ratio < 1 is certified by

  prefix_penalty < total_gain.

This file formalizes the log-domain algebra.  The real-analysis facts about
log and exp are standard; using this log form avoids a heavy Mathlib cache
dependency while still machine-checking the decision rule used by the notebooks.
-/

def logPredictedRatio (prefixPenalty totalGain : Int) : Int :=
  prefixPenalty - totalGain

theorem log_tradeoff_certificate
    {prefixPenalty totalGain : Int}
    (h : prefixPenalty < totalGain) :
    logPredictedRatio prefixPenalty totalGain < 0 := by
  unfold logPredictedRatio
  exact Int.sub_neg_of_lt h

def accumulatedGain3 (g1 g2 g3 : Int) : Int :=
  g1 + g2 + g3

theorem accumulated_log_tradeoff_certificate
    {prefixPenalty g1 g2 g3 : Int}
    (h : prefixPenalty < accumulatedGain3 g1 g2 g3) :
    logPredictedRatio prefixPenalty (accumulatedGain3 g1 g2 g3) < 0 := by
  exact log_tradeoff_certificate h

def firstOrderLossRatio (rho : Int) : Int :=
  1 - 2 * rho

theorem positive_rate_improves_first_order_loss
    {rho : Int}
    (hrho : 0 < rho) :
    firstOrderLossRatio rho < 1 := by
  unfold firstOrderLossRatio
  omega

def pointwiseImproves {n : Nat} (penalty gain : Fin n -> Int) : Prop :=
  forall t : Fin n, logPredictedRatio (penalty t) (gain t) < 0

theorem adjustment_method_pointwise_improves
    {n : Nat}
    {penalty gain : Fin n -> Int}
    (hcert : forall t : Fin n, penalty t < gain t) :
    pointwiseImproves penalty gain := by
  intro t
  exact log_tradeoff_certificate (hcert t)

```

## 如何使用这个证书

对两个调度，定义

$$
\text{prefix penalty}
=
\log\frac{L(b)}{L(a)},
\qquad
\text{future gain}
=
c\bigl(\lambda(b)-\lambda(a)\bigr).
$$

如果

$$
\text{future gain}>\text{prefix penalty},
$$

那么定理保证：在局部指数模型下，调度 $b$ 的预测终点 loss 小于调度 $a$。

这并不证明真实非线性训练轨迹一定永远符合预测。定理证明的是：在清楚写出的模型假设下，这个决策规则是严格成立的。实验 notebook 则用于检验这个模型在什么时候足够准确、足够有用。